In [ ]:
pip install transformers torch datasets scikit-learn pandas

**Loading The dataset**

In [ ]:
from datasets import load_dataset
import pandas as pd

df = load_dataset('Tobi-Bueck/customer-support-tickets')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…):   0%|          | 0.00/26.0M [00:00<?, ?B/s]

(…)set-tickets-german_normalized_50_5_2.csv: 0.00B [00:00, ?B/s]

dataset-tickets-multi-lang-4-20k.csv:   0%|          | 0.00/18.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

In [ ]:
df = pd.DataFrame(df['train'])
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


In [ ]:
print(df.columns)

Index(['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language',
       'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6',
       'tag_7', 'tag_8'],
      dtype='object')


In [ ]:
# target label
df['text'] = df['subject'] + ' ' + df['body']
df['queue'].value_counts()


,count
queue,
Technical Support,14186
Product Support,8960
Customer Service,7420
IT Support,5725
Billing and Payments,4874
Returns and Exchanges,2438
Service Outages and Maintenance,1912
Sales and Pre-Sales,1490
Human Resources,914


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['category'] = le.fit_transform(df['queue'])

In [ ]:
df['category'].unique()

array([49, 41,  6, 42, 45, 39, 28, 10, 23, 16, 13,  7, 22, 24, 33, 47, 34,
       51,  8, 21, 19, 30,  9, 32,  5, 12, 27, 37, 50,  4, 35, 43, 26,  0,
       36, 46, 44, 11, 17, 18,  2, 29, 40, 14,  3, 20, 31, 15, 25, 48,  1,
       38])

**Train test data splitting**

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category'])

**Tokenization**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
  return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=128)



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
test_df['text'].head()

,text
8295,Fehler in der Datenpipeline Die Datenanalyse-P...
10202,Report on Data Security Incident at the Hospit...
10022,"Hilfe bei digitalen Strategien Ich schreibe, u..."
46641,Probleme bei der Anmeldung auf unserer Website...
20357,Ensuring Security in Medical Data Services Cus...


**Dataset Formatting**

In [ ]:
from datasets import Dataset

train_df['text'] = train_df['subject'].fillna('') + ' ' + train_df['body'].fillna('')
test_df['text'] = test_df['subject'].fillna('') + ' ' + test_df['body'].fillna('')

train_dataset = Dataset.from_pandas(train_df[['text','category']],preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[['text','category']],preserve_index=False)

train_dataset = train_dataset.map(tokenize, batched=True, batch_size=len(train_dataset))
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=len(test_dataset))

train_dataset = train_dataset.rename_columns({'category':'labels'})
test_dataset = test_dataset.rename_columns({'category':'labels'})

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/49412 [00:00<?, ? examples/s]

Map:   0%|          | 0/12353 [00:00<?, ? examples/s]

**Model Loading**

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(le.classes_))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Training & Evaluation**

In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score,f1_score

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='no',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    save_strategy='no'

)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted')
    }


trainer  = Trainer(model=model, args=training_args, train_dataset=train_dataset.select(range(30000)), eval_dataset=test_dataset.select(range(12000)),compute_metrics=compute_metrics)
trainer.train()

Step,Training Loss
500,2.185538
1000,2.128727
1500,2.117456
2000,2.013416
2500,1.897798
3000,1.895655
3500,1.781478
4000,1.776211
4500,1.718082
5000,1.682967


TrainOutput(global_step=15000, training_loss=1.5434463216145833, metrics={'train_runtime': 1839.7032, 'train_samples_per_second': 32.614, 'train_steps_per_second': 8.153, 'total_flos': 3948437606400000.0, 'train_loss': 1.5434463216145833, 'epoch': 2.0})

**Evaluation**

In [ ]:
import torch
def predict_ticket(text):
    inputs = tokenizer(text, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    # Move inputs to the same device as the model
    device = model.device # Get the device of the model
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move all input tensors to the device

    with torch.no_grad():
        logits = model(**inputs).logits
    prediction = torch.argmax(logits, dim=1).item()
    return prediction

In [ ]:
print(predict_ticket("my account is not working"))
print(predict_ticket("payment related issue"))
print(predict_ticket("i want to cancel my subscription"))
print(predict_ticket("damaged package got"))

6
6
41
45


In [ ]:
res = trainer.evaluate()
print(res)

{'eval_loss': 1.3696997165679932, 'eval_accuracy': 0.54175, 'eval_f1': 0.5259145694968507, 'eval_runtime': 96.9553, 'eval_samples_per_second': 123.768, 'eval_steps_per_second': 30.942, 'epoch': 2.0}


In [ ]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)

{'Arts & Entertainment/Movies': np.int64(0), 'Arts & Entertainment/Music': np.int64(1), 'Autos & Vehicles/Maintenance': np.int64(2), 'Autos & Vehicles/Sales': np.int64(3), 'Beauty & Fitness/Cosmetics': np.int64(4), 'Beauty & Fitness/Fitness Training': np.int64(5), 'Billing and Payments': np.int64(6), 'Books & Literature/Fiction': np.int64(7), 'Books & Literature/Non-Fiction': np.int64(8), 'Business & Industrial/Manufacturing': np.int64(9), 'Customer Service': np.int64(10), 'Finance/Investments': np.int64(11), 'Finance/Personal Finance': np.int64(12), 'Food & Drink/Groceries': np.int64(13), 'Food & Drink/Restaurants': np.int64(14), 'Games': np.int64(15), 'General Inquiry': np.int64(16), 'Health/Medical Services': np.int64(17), 'Health/Mental Health': np.int64(18), 'Hobbies & Leisure/Collectibles': np.int64(19), 'Hobbies & Leisure/Crafts': np.int64(20), 'Home & Garden/Home Improvement': np.int64(21), 'Home & Garden/Landscaping': np.int64(22), 'Human Resources': np.int64(23), 'IT & Techno